# Análisis Exploratorio de Datos (EDA)
## Dataset: Anuncios de Venta de Vehículos en EE. UU.

Este notebook realiza un análisis exploratorio básico del conjunto de datos `vehicles_us.csv`,
que contiene anuncios de venta de vehículos usados en los Estados Unidos.

**Columnas disponibles:**
- `price` – Precio del vehículo (USD)
- `model_year` – Año del modelo
- `model` – Modelo del vehículo
- `condition` – Condición (new, like new, excellent, good, fair, salvage)
- `cylinders` – Número de cilindros
- `fuel` – Tipo de combustible
- `odometer` – Kilómetros/millas recorridas
- `transmission` – Tipo de transmisión
- `type` – Tipo de vehículo (SUV, sedan, pickup, etc.)
- `paint_color` – Color de la pintura
- `is_4wd` – Si tiene tracción en las 4 ruedas
- `date_posted` – Fecha de publicación
- `days_listed` – Días en el mercado

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Cargar el dataset
df = pd.read_csv('../vehicles_us.csv')
print(f'Shape del dataset: {df.shape}')
df.head()

## 1. Información general del dataset

In [ ]:
# Información básica
df.info()

In [ ]:
# Estadísticas descriptivas
df.describe()

In [ ]:
# Valores nulos por columna
nulls = df.isnull().sum().reset_index()
nulls.columns = ['columna', 'valores_nulos']
nulls['porcentaje'] = (nulls['valores_nulos'] / len(df) * 100).round(2)
nulls[nulls['valores_nulos'] > 0]

## 2. Distribución del Odómetro (Histograma)

In [ ]:
# Histograma del odómetro
fig = px.histogram(
    df,
    x='odometer',
    nbins=50,
    title='Distribución del Odómetro',
    labels={'odometer': 'Odómetro (millas)', 'count': 'Cantidad'},
    color_discrete_sequence=['#1f77b4']
)
fig.update_layout(bargap=0.05)
fig.show()

In [ ]:
# Histograma del precio (filtrando outliers)
df_price = df[(df['price'] > 500) & (df['price'] < 100_000)]

fig2 = px.histogram(
    df_price,
    x='price',
    nbins=60,
    title='Distribución del Precio de Venta',
    labels={'price': 'Precio (USD)', 'count': 'Cantidad'},
    color_discrete_sequence=['#2ca02c']
)
fig2.show()

## 3. Gráfico de Dispersión: Precio vs Odómetro

In [ ]:
# Filtrar valores extremos
df_clean = df[
    (df['price'] > 500) & (df['price'] < 100_000) &
    (df['odometer'] > 0) & (df['odometer'] < 400_000)
]

# Gráfico de dispersión: Precio vs Odómetro
fig3 = px.scatter(
    df_clean,
    x='odometer',
    y='price',
    color='condition',
    title='Precio vs Odómetro por Condición del Vehículo',
    labels={
        'odometer': 'Odómetro (millas)',
        'price': 'Precio (USD)',
        'condition': 'Condición'
    },
    opacity=0.5,
    hover_data=['model', 'model_year']
)
fig3.show()

## 4. Análisis por categorías

In [ ]:
# Precio promedio por tipo de vehículo
avg_by_type = (
    df.groupby('type')['price']
    .mean()
    .reset_index()
    .sort_values('price', ascending=False)
)

fig4 = px.bar(
    avg_by_type,
    x='type',
    y='price',
    title='Precio Promedio por Tipo de Vehículo',
    labels={'type': 'Tipo', 'price': 'Precio Promedio (USD)'},
    color='price',
    color_continuous_scale='Blues'
)
fig4.show()

In [ ]:
# Distribución por tipo de combustible
fuel_counts = df['fuel'].value_counts().reset_index()
fuel_counts.columns = ['fuel', 'count']

fig5 = px.pie(
    fuel_counts,
    names='fuel',
    values='count',
    title='Distribución por Tipo de Combustible'
)
fig5.show()

In [ ]:
# Días en el mercado vs precio
fig6 = px.scatter(
    df_clean,
    x='days_listed',
    y='price',
    color='type',
    title='Días en el Mercado vs Precio',
    labels={
        'days_listed': 'Días en el mercado',
        'price': 'Precio (USD)',
        'type': 'Tipo de vehículo'
    },
    opacity=0.5
)
fig6.show()

## 5. Conclusiones del EDA

- **Odómetro**: La mayoría de los vehículos tienen entre 50,000 y 200,000 millas recorridas.
- **Precio**: La distribución de precios está sesgada hacia la derecha; la mayoría de los vehículos se venden por menos de $20,000.
- **Condición vs Precio**: Los vehículos en mejores condiciones tienden a tener precios más altos y menos millas.
- **Tipo de combustible**: El combustible de gasolina domina el mercado.
- **Tiempo de venta**: No hay una correlación clara entre el precio y los días listados.